In [ ]:
import wrds
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
from datetime import datetime


db = wrds.Connection() 

In [ ]:
def plot_financial_engine(df, idx):
    fig, axes = plt.subplots(4, 1, figsize=(12, 22))
    
    
    plot_configs = [
        (['roa', 'roe', 'roc'], 'Profitability Metrics (ROA/ROE/ROC)', 0),
        (['current_ratio', 'quick_ratio'], 'Liquidity Analysis (Current/Quick)', 1),
        (['roa_ma'], 'ROA 4-Quarter Moving Average (Trend)', 2),
        (['ret'], 'Quarterly Returns (%)', 3)
    ]
    
    for cols, title, i in plot_configs:
        for col in cols:
            axes[i].plot(df['datadate'], df[col], label=col.upper(), marker='o', markersize=4, alpha=0.7)
            
            axes[i].scatter(df.loc[idx, 'datadate'], df.loc[idx, col], 
                            color='red', s=130, edgecolors='black', zorder=10)
        
        
        if i == 0:
            axes[i].axhline(y=0.05, color='gray', linestyle='--', label='Industry Baseline (5%)')
            
        axes[i].set_title(title, fontsize=14, fontweight='bold')
        axes[i].legend()
        axes[i].grid(True, linestyle=':', alpha=0.6)
    
    plt.tight_layout()
    plt.show()


def calculate_risk_metrics(df, idx):
    recent_rets = df.loc[idx:, 'ret'].dropna()
    if len(recent_rets) < 2:
        print("提示：选定日期后数据点不足，无法计算风险指标。")
        return
        
    
    vol = recent_rets.std() * np.sqrt(4)
    sharpe = (recent_rets.mean() / recent_rets.std()) * np.sqrt(4) if recent_rets.std() != 0 else 0
    
    print(f"\n--- 📈 风险与专业指标 (自选定日期起) ---")
    print(f"📊 年化波动率: {vol:.2%}")
    print(f"⚖️ 风险调整后收益 (Sharpe Ratio): {sharpe:.2f}")
    print("---------------------------------------")


In [ ]:
ticker_input = widgets.Text(value='NVDA', description='美股代码:', placeholder='例如: TSLA')
date_input = widgets.DatePicker(
    description='选择日期:',
    value=datetime(2024, 4, 8).date(),
    min=datetime(2021, 1, 1).date(),
    max=datetime(2026, 1, 1).date()
)
btn = widgets.Button(description="生成深度价值报告", button_style='danger')
out = widgets.Output()

def on_click_analyze(b):
    with out:
        clear_output(wait=True)
        ticker = ticker_input.value.strip().upper()
        sel_date = pd.to_datetime(date_input.value).normalize()
        
        
        query = f"""
        SELECT datadate, prccq,
               (niq / NULLIF(atq, 0)) as roa,
               (niq / NULLIF(ceqq, 0)) as roe,
               (oiadpq / NULLIF(atq - lctq, 0)) as roc,
               (actq / NULLIF(lctq, 0)) as current_ratio,
               ((cheq + rectq) / NULLIF(lctq, 0)) as quick_ratio
        FROM comp.fundq
        WHERE tic = '{ticker}' AND datadate >= '2020-01-01' AND datadate <= '2026-01-01'
        ORDER BY datadate
        """
        
        try:
            df = db.raw_sql(query)
            if df is None or df.empty:
                print(f"❌ 未找到 {ticker} 的相关财务数据。")
                return
                
            
            df['datadate'] = pd.to_datetime(df['datadate'])
            df['roa_ma'] = df['roa'].rolling(window=4).mean() 
            df['ret'] = df['prccq'].pct_change()            
            
            
            idx = (df['datadate'] - sel_date).abs().idxmin()
            latest = df.loc[idx]
            
            
            print(f"🚀 Alpha Insight Engine | 报告对象: {ticker}")
            print(f"📅 选定季度节点: {latest['datadate'].date()}")
            print("-" * 40)
            
            
            if latest['current_ratio'] < 1.2:
                print("🔴 偿债风险提示：Current Ratio 低于 1.2，请关注短期现金流。")
            else:
                print("🟢 财务健康状况：流动性充足，短期偿债能力良好。")
                
            
            post_data = df.loc[idx:]
            if len(post_data) > 1:
                total_gain = (post_data['prccq'].iloc[-1] / latest['prccq'] - 1) * 100
                print(f"💰 价值模拟：若在该点入场并持有至今，总收益率为 {total_gain:.2f}%")
            
            
            plot_financial_engine(df, idx)
            calculate_risk_metrics(df, idx)
            
        except Exception as e:
            print(f"⚠️ 分析过程中出现错误: {e}")

btn.on_click(on_click_analyze)
display(widgets.VBox([ticker_input, date_input, btn]), out)


In [ ]:
db.close()